# LeetCode 814 — Binary Tree Pruning bằng Python

Link đề: https://leetcode.com/problems/binary-tree-pruning/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        ...
```

Ghi chú: cây nhị phân chỉ gồm nút `0` và `1`. Ta phải **cắt bỏ mọi cây con không chứa số `1` nào**. Điểm mấu chốt: muốn biết một nút có phải cắt hay không, phải **biết kết quả cắt ở hai con trước** (duyệt **hậu thứ tự**, postorder). Notebook tự dựng `TreeNode`, hàm **dựng cây từ danh sách kiểu LeetCode** (`[1,null,0,0,1]`) và hàm **xuất cây ra danh sách** để chạy thử và so sánh.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trả về **cùng một cây** sau khi loại bỏ mọi cây con (một nút và toàn bộ hậu duệ của nó) **không chứa giá trị `1`**.
2. Nắm các công cụ Python cho cây: lớp `TreeNode`, hàm dựng cây từ danh sách theo thứ tự tầng (level-order, dùng `None` cho nút vắng), hàm xuất cây ra danh sách, đệ quy, ngăn xếp, hàng đợi `deque`, và so sánh `None` bằng `is`.
3. Đi từ **kiểm tra lại "có số 1 không" cho từng nút (`O(n^2)`)** → **đệ quy hậu thứ tự (`O(n)`)** → **hậu thứ tự lặp bằng ngăn xếp** → **duyệt tầng ngược và tự gỡ khỏi cha** → **dựng cây mới (không sửa cây gốc)**.
4. Hiểu mấu chốt: một nút bị cắt khi **giá trị là `0` và cả hai con (sau khi cắt) đều rỗng**; phải **cắt hai con trước, quyết định nút sau** (hậu thứ tự) để cắt được dây chuyền các nút `0` nối dài.
5. Tránh các lỗi kinh điển: quyết định cắt trước khi cắt các con (tiền thứ tự), gọi đệ quy mà quên gán lại `root.left = ...`, xóa nút `0` dù con còn chứa `1`, chỉ cắt lá một lượt, và quên trả về gốc mới (gốc cũng có thể bị cắt).

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân. Trả về **cây đó** sau khi **mọi cây con không chứa `1` đã bị loại bỏ**.

Cây con của nút `node` là `node` cộng với mọi hậu duệ của nó.

**Ví dụ 1**

- Input: `root = [1,null,0,0,1]`
- Output: `[1,null,0,null,1]`
- Giải thích: nút `0` ở đáy trái (lá `0`) không chứa `1` nên bị cắt; nút `0` bên phải gốc có con phải là `1` nên được giữ.

**Ví dụ 2**

- Input: `root = [1,0,1,0,0,0,1]`
- Output: `[1,null,1,null,1]`

**Ví dụ 3**

- Input: `root = [1,1,0,1,1,0,1,0]`
- Output: `[1,1,0,1,1,null,1]`

(Danh sách theo kiểu LeetCode: duyệt theo tầng, `null` (ở đây là `None`) cho nút vắng, bỏ các `null` thừa ở cuối.)

**Ràng buộc:**

- Số nút trong cây thuộc đoạn `[1, 200]`.
- `Node.val` là `0` hoặc `1`.

Chú ý: nếu **cả cây** không có số `1` nào thì kết quả là **cây rỗng** (`None`, in ra `[]`).

In [1]:
from typing import List, Optional
from collections import deque
import random
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ danh sách theo thứ tự tầng kiểu LeetCode (None = nút vắng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_list(root):
    """Xuất cây ra danh sách theo thứ tự tầng kiểu LeetCode, bỏ None thừa ở cuối."""
    if root is None:
        return []
    out = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, None, 0, 0, 1], [1, None, 0, None, 1]),
    ([1, 0, 1, 0, 0, 0, 1], [1, None, 1, None, 1]),
    ([1, 1, 0, 1, 1, 0, 1, 0], [1, 1, 0, 1, 1, None, 1]),
]
for values, expected in examples:
    root = build_tree(values)
    assert to_list(root) == values          # dựng rồi xuất phải ra đúng danh sách ban đầu
    print(values, "->", to_list(root), "| kỳ vọng sau cắt:", expected)

[1, None, 0, 0, 1] -> [1, None, 0, 0, 1] | kỳ vọng sau cắt: [1, None, 0, None, 1]
[1, 0, 1, 0, 0, 0, 1] -> [1, 0, 1, 0, 0, 0, 1] | kỳ vọng sau cắt: [1, None, 1, None, 1]
[1, 1, 0, 1, 1, 0, 1, 0] -> [1, 1, 0, 1, 1, 0, 1, 0] | kỳ vọng sau cắt: [1, 1, 0, 1, 1, None, 1]


## 3. Kiến thức nền

### 3.1. Nút cây `TreeNode` và con trỏ `None`

Mỗi nút có `val`, `left`, `right`. `left`/`right` là **nút con** hoặc `None` (không có con). Một cây được xác định bởi nút **gốc**; cả cây rỗng là `None`. Ta kiểm tra "không có con" bằng `node.left is None` (nên dùng `is`, không phải `==`).

In [2]:
root = TreeNode(1, TreeNode(0), TreeNode(1))
print(root.val, root.left.val, root.right.val)
print(root.left.left is None, root.left.right is None)
print(to_list(root))

1 0 1
True True
[1, 0, 1]


### 3.2. Dựng cây từ danh sách theo thứ tự tầng (kiểu LeetCode)

Danh sách `[1, None, 0, 0, 1]` mô tả cây theo từng tầng: phần tử `0` là gốc; với mỗi nút (theo thứ tự đã dựng) hai phần tử kế tiếp là con trái và con phải (`None` nghĩa là vắng, và nút vắng **không có con** nên không chiếm chỗ trong danh sách). Dùng **hàng đợi** `deque`: lấy nút đầu hàng, gán hai phần tử kế làm con, đẩy các con thật vào hàng. `to_list` làm ngược lại. Dưới đây vẽ cây dạng chữ để dễ nhìn.

In [3]:
def draw(root, indent="", tag="gốc"):
    if root is None:
        return
    print(f"{indent}[{tag}] {root.val}")
    if root.left is not None or root.right is not None:
        draw(root.left, indent + "    ", "trái") if root.left else print(f"{indent}    [trái] -")
        draw(root.right, indent + "    ", "phải") if root.right else print(f"{indent}    [phải] -")

draw(build_tree([1, None, 0, 0, 1]))
print(to_list(build_tree([1, None, 0, 0, 1])))

[gốc] 1
    [trái] -
    [phải] 0
        [trái] 0
        [phải] 1
[1, None, 0, 0, 1]


### 3.3. Đệ quy trên cây: tiền thứ tự và hậu thứ tự

Một hàm đệ quy trên cây có ba chỗ có thể "làm việc": **trước** khi xuống hai con (tiền thứ tự, preorder), **giữa** hai con, hoặc **sau** khi cả hai con xong (hậu thứ tự, postorder). Muốn biết "cây con của tôi có chứa `1` không" thì phải biết kết quả của **hai con trước**: nên làm việc ở **sau** (hậu thứ tự). Ví dụ đếm số nút bằng hậu thứ tự:

In [4]:
def count_nodes(node):
    if node is None:
        return 0
    left = count_nodes(node.left)        # xuống con trái trước
    right = count_nodes(node.right)      # rồi con phải
    return left + right + 1              # làm việc SAU khi biết cả hai con

print(count_nodes(build_tree([1, 1, 0, 1, 1, 0, 1, 0])))

8


### 3.4. Hàm trả về nút để "gán lại" con

Mẹo hay dùng khi sửa cây bằng đệ quy: hàm `f(node)` **trả về gốc mới** của cây con sau khi xử lý (có thể là `node`, hoặc `None` nếu cây con bị xóa). Cha gán lại: `node.left = f(node.left)`. Nhờ vậy xóa một cây con chỉ là **trả về `None`** và để cha gán.

In [5]:
def remove_value(node, target):
    """Ví dụ: xóa mọi LÁ có giá trị target (một lượt, không dây chuyền)."""
    if node is None:
        return None
    node.left = remove_value(node.left, target)     # gán lại con trái
    node.right = remove_value(node.right, target)   # gán lại con phải
    if node.left is None and node.right is None and node.val == target:
        return None                                  # xóa nút này: trả None cho cha gán
    return node

print(to_list(remove_value(build_tree([1, 0, 0, 0]), 0)))

[1]


### 3.5. Ngăn xếp (list) và `deque`

Ngăn xếp (stack) là danh sách dùng `append` (đẩy) và `pop()` (lấy phần tử cuối): **vào sau ra trước**. `deque` là hàng đợi hai đầu: `popleft()` lấy phần tử đầu: **vào trước ra trước** (BFS). Hai cấu trúc này giúp duyệt cây **không dùng đệ quy** (cách 3 và 4).

In [6]:
stack = []
stack.append("a"); stack.append("b")
print(stack.pop(), stack.pop())          # b a (vào sau ra trước)
q = deque(["a", "b"])
q.append("c")
print(q.popleft(), q.popleft())          # a b (vào trước ra trước)

b a
a b


### 3.6. Tập hợp các nút đã bị xóa

Nút (đối tượng `TreeNode`) băm theo **danh tính**, nên đặt được vào `set`. Cách 3 dùng `dead = set()` ghi nhớ các nút đã bị cắt: khi cha xử lý, nó kiểm tra `node.left in dead`.

In [7]:
a, b = TreeNode(0), TreeNode(0)
dead = {a}
print(a in dead, b in dead)
print(TreeNode(0) == TreeNode(0))        # hai nút khác nhau dù cùng giá trị

True False
False


## 4. Ý tưởng nền tảng / trực giác

Vẽ cây của ví dụ 1: `[1, None, 0, 0, 1]`.

In [8]:
tree = build_tree([1, None, 0, 0, 1])
draw(tree)

[gốc] 1
    [trái] -
    [phải] 0
        [trái] 0
        [phải] 1


Xét từng cây con (từ **dưới lên**):

- Lá `0` (con trái của nút `0`): cây con chỉ có nút `0`, **không chứa `1`** → cắt.
- Lá `1` (con phải của nút `0`): chứa `1` → giữ.
- Nút `0` giữa (có con `0` đã cắt và con `1`): cây con còn chứa `1` (qua lá `1`) → giữ.
- Gốc `1`: chứa `1` → giữ.

Tức là **một nút bị cắt khi cả cây con của nó không có `1`**. Với nút `v` có hai con (đã xử lý xong): `v` bị cắt khi `v.val == 0` **và** cả hai con đều bị cắt (rỗng). Thứ tự bắt buộc là **dưới lên**: phải biết hai con còn tồn tại hay không rồi mới quyết định cho `v`. Ví dụ: một dây chuyền `0 -> 0 -> 0` cuối cùng đều bị cắt, nhưng nút `0` trên cùng chỉ biết nó phải cắt **sau khi** các nút `0` dưới nó đã bị cắt.

Hãy thử tay một cây lớn hơn bằng cách kiểm tra từng nút "cây con của nó có chứa `1` không?":

In [9]:
def contains_one(node):
    if node is None:
        return False
    return node.val == 1 or contains_one(node.left) or contains_one(node.right)

def walk_report(node, path="gốc"):
    if node is None:
        return
    print(f"nút {path:<22} val={node.val}  cây con chứa 1? {contains_one(node)}")
    walk_report(node.left, path + "->trái")
    walk_report(node.right, path + "->phải")

walk_report(build_tree([1, 0, 1, 0, 0, 0, 1]))

nút gốc                    val=1  cây con chứa 1? True
nút gốc->trái              val=0  cây con chứa 1? False
nút gốc->trái->trái        val=0  cây con chứa 1? False
nút gốc->trái->phải        val=0  cây con chứa 1? False
nút gốc->phải              val=1  cây con chứa 1? True
nút gốc->phải->trái        val=0  cây con chứa 1? False
nút gốc->phải->phải        val=1  cây con chứa 1? True


Những nút có "chứa 1? False" chính là các nút cần cắt (và vì cả cây con của chúng không có `1`, ta cắt chúng cùng toàn bộ hậu duệ). Ví dụ 2 cho kết quả `[1, None, 1, None, 1]`: nút `0` bên trái gốc với hai con `0` đều bị cắt.

## 5. Cách 1 — Brute Force: kiểm tra "có số 1" lại cho từng nút

**Ý tưởng.** Với mỗi nút, **duyệt cả cây con** của nó bằng `contains_one` để biết nó có chứa `1` không. Nếu không chứa → trả `None` (cắt cả cây con). Nếu có → giữ nút và xử lý đệ quy hai con theo cùng cách.

In [10]:
def contains_one_nodes(node):
    if node is None:
        return False
    return node.val == 1 or contains_one_nodes(node.left) or contains_one_nodes(node.right)


def prune_brute(node):
    if node is None:
        return None
    if not contains_one_nodes(node):         # duyệt cả cây con chỉ để trả lời câu hỏi này
        return None
    node.left = prune_brute(node.left)
    node.right = prune_brute(node.right)
    return node


for values, expected in examples:
    got = to_list(prune_brute(build_tree(values)))
    print(values, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, None, 0, 0, 1] -> [1, None, 0, None, 1] (kỳ vọng [1, None, 0, None, 1] )
[1, 0, 1, 0, 0, 0, 1] -> [1, None, 1, None, 1] (kỳ vọng [1, None, 1, None, 1] )
[1, 1, 0, 1, 1, 0, 1, 0] -> [1, 1, 0, 1, 1, None, 1] (kỳ vọng [1, 1, 0, 1, 1, None, 1] )


**Độ phức tạp:** Time `O(n^2)` trong trường hợp xấu (cây là một dây chuyền dài: nút ở độ sâu `d` có cây con `n - d` nút và ta duyệt lại mỗi lần). Space `O(h)` cho ngăn xếp đệ quy với `h` là chiều cao.

**Nút thắt:** câu hỏi "cây con của nút này có `1` không" bị **trả lời lặp lại** ở nhiều nút (mỗi nút duyệt lại cả cây con của nó). Nếu hai con đã được xử lý xong thì ta **biết ngay** câu trả lời cho nút cha: đó là bước đột phá ở mục 6.

## 6. Ý tưởng thuật toán / bước đột phá

Hàm `prune(node)` trả về **gốc của cây con sau khi cắt** (hoặc `None` nếu cả cây con bị cắt). Hậu thứ tự:

1. `node is None` → trả `None`.
2. `node.left = prune(node.left)`, `node.right = prune(node.right)` (cắt hai con trước).
3. Nếu `node.val == 0` **và** `node.left is None` **và** `node.right is None` → nút này giờ là một lá `0`, cây con không chứa `1` → trả `None`.
4. Ngược lại trả `node`.

**Vì sao đủ?** Sau bước 2, cây con của `node` chỉ còn các nút mà cây con của chúng chứa `1`. Do đó cây con của `node` chứa `1` khi và chỉ khi `node.val == 1`, **hoặc** còn ít nhất một con (con còn lại thì cây con của nó có `1`). Vậy `node` bị cắt đúng khi `node.val == 0` và hai con đều rỗng. Mỗi nút xử lý **một lần**: `O(n)`.

## 7. Các cách tiếp theo

Phần này có thêm 4 cách nữa (tổng 5 cách): đệ quy hậu thứ tự (cách nên nộp), hậu thứ tự lặp bằng ngăn xếp, duyệt tầng ngược rồi tự gỡ khỏi cha, và dựng cây mới không sửa cây gốc.

### 7.1. Cách 2 — Đệ quy hậu thứ tự (cách nên nộp)

**Giải thích từng đoạn.**

1. `if root is None: return None`.
2. `root.left = self.pruneTree(root.left)`; `root.right = self.pruneTree(root.right)`: cắt hai con **và gán lại** (con bị cắt trả `None`).
3. `if root.val == 0 and root.left is None and root.right is None: return None`.
4. `return root`.

In [11]:
def prune_recursive(root):
    if root is None:
        return None
    root.left = prune_recursive(root.left)       # cắt con trái, gán lại
    root.right = prune_recursive(root.right)     # cắt con phải, gán lại
    if root.val == 0 and root.left is None and root.right is None:
        return None                              # lá 0 sau khi cắt: bỏ luôn
    return root


for values, expected in examples:
    got = to_list(prune_recursive(build_tree(values)))
    print(values, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, None, 0, 0, 1] -> [1, None, 0, None, 1] (kỳ vọng [1, None, 0, None, 1] )
[1, 0, 1, 0, 0, 0, 1] -> [1, None, 1, None, 1] (kỳ vọng [1, None, 1, None, 1] )
[1, 1, 0, 1, 1, 0, 1, 0] -> [1, 1, 0, 1, 1, None, 1] (kỳ vọng [1, 1, 0, 1, 1, None, 1] )


In [12]:
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left = self.pruneTree(root.left)
        root.right = self.pruneTree(root.right)
        if root.val == 0 and root.left is None and root.right is None:
            return None
        return root

print([to_list(Solution().pruneTree(build_tree(v))) for v, _ in examples])

[[1, None, 0, None, 1], [1, None, 1, None, 1], [1, 1, 0, 1, 1, None, 1]]


**Độ phức tạp:** Time `O(n)` (mỗi nút thăm đúng một lần). Space `O(h)` cho ngăn xếp đệ quy, tối đa `O(n)` khi cây là một dây chuyền (với `n ≤ 200` an toàn, không vượt giới hạn đệ quy 1000).

### 7.2. Cách 3 — Hậu thứ tự lặp bằng ngăn xếp (không đệ quy)

**Giải thích từng đoạn.** Mô phỏng hậu thứ tự bằng ngăn xếp các cặp `(nút, đã_thăm_con_chưa)`. Lần đầu lấy nút ra (chưa thăm con): đẩy lại `(nút, True)`, rồi đẩy hai con (con phải trước, con trái sau để con trái được lấy ra trước). Lần hai lấy nút ra (`True`) thì hai con đã xử lý xong: nếu con nào nằm trong `dead` (đã bị cắt) thì gỡ (`node.left = None`); nếu `node.val == 0` và không còn con thì thêm `node` vào `dead`. Cuối cùng: gốc nằm trong `dead` thì trả `None`.

In [13]:
def prune_iterative(root):
    if root is None:
        return None
    dead = set()
    stack = [(root, False)]
    while stack:
        node, children_done = stack.pop()
        if not children_done:
            stack.append((node, True))               # quay lại sau khi hai con xong
            if node.right is not None:
                stack.append((node.right, False))
            if node.left is not None:
                stack.append((node.left, False))
        else:
            if node.left in dead:
                node.left = None
            if node.right in dead:
                node.right = None
            if node.val == 0 and node.left is None and node.right is None:
                dead.add(node)
    return None if root in dead else root


for values, expected in examples:
    got = to_list(prune_iterative(build_tree(values)))
    print(values, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, None, 0, 0, 1] -> [1, None, 0, None, 1] (kỳ vọng [1, None, 0, None, 1] )
[1, 0, 1, 0, 0, 0, 1] -> [1, None, 1, None, 1] (kỳ vọng [1, None, 1, None, 1] )
[1, 1, 0, 1, 1, 0, 1, 0] -> [1, 1, 0, 1, 1, None, 1] (kỳ vọng [1, 1, 0, 1, 1, None, 1] )


In [14]:
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        dead = set()
        stack = [(root, False)]
        while stack:
            node, children_done = stack.pop()
            if not children_done:
                stack.append((node, True))
                if node.right is not None:
                    stack.append((node.right, False))
                if node.left is not None:
                    stack.append((node.left, False))
            else:
                if node.left in dead:
                    node.left = None
                if node.right in dead:
                    node.right = None
                if node.val == 0 and node.left is None and node.right is None:
                    dead.add(node)
        return None if root in dead else root

print(to_list(Solution().pruneTree(build_tree([1, None, 0, 0, 1]))))

[1, None, 0, None, 1]


**Độ phức tạp:** Time `O(n)`; Space `O(n)` cho ngăn xếp, `dead` và các cặp. Ưu điểm: không bị giới hạn đệ quy với cây rất sâu.

### 7.3. Cách 4 — Duyệt tầng ngược: nút sâu xử lý trước, tự gỡ khỏi cha

**Giải thích từng đoạn.** BFS lập danh sách `order` các nút theo thứ tự tầng và `parent[nút] = (cha, 'L' hoặc 'R')`. Duyệt `order` **ngược**: mọi nút sâu hơn được xử lý trước nút nông hơn nên khi tới một nút, hai con (nếu còn) đã được xử lý. Nếu `nút.val == 0` và không còn con thì **gỡ nó khỏi cha** (`cha.left = None` hoặc `cha.right = None`); nếu nút là gốc thì trả `None`.

In [15]:
def prune_reverse_bfs(root):
    if root is None:
        return None
    order = []
    parent = {}                                       # nút -> (cha, 'L' / 'R')
    queue = deque([root])
    while queue:
        node = queue.popleft()
        order.append(node)
        if node.left is not None:
            parent[node.left] = (node, "L")
            queue.append(node.left)
        if node.right is not None:
            parent[node.right] = (node, "R")
            queue.append(node.right)
    for node in reversed(order):                      # sâu trước, nông sau
        if node.val == 0 and node.left is None and node.right is None:
            if node is root:
                return None
            par, side = parent[node]
            if side == "L":
                par.left = None
            else:
                par.right = None
    return root


for values, expected in examples:
    got = to_list(prune_reverse_bfs(build_tree(values)))
    print(values, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, None, 0, 0, 1] -> [1, None, 0, None, 1] (kỳ vọng [1, None, 0, None, 1] )
[1, 0, 1, 0, 0, 0, 1] -> [1, None, 1, None, 1] (kỳ vọng [1, None, 1, None, 1] )
[1, 1, 0, 1, 1, 0, 1, 0] -> [1, 1, 0, 1, 1, None, 1] (kỳ vọng [1, 1, 0, 1, 1, None, 1] )


In [16]:
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        order, parent = [], {}
        queue = deque([root])
        while queue:
            node = queue.popleft()
            order.append(node)
            for child, side in ((node.left, "L"), (node.right, "R")):
                if child is not None:
                    parent[child] = (node, side)
                    queue.append(child)
        for node in reversed(order):
            if node.val == 0 and node.left is None and node.right is None:
                if node is root:
                    return None
                par, side = parent[node]
                if side == "L":
                    par.left = None
                else:
                    par.right = None
        return root

print(to_list(Solution().pruneTree(build_tree([1, 0, 1, 0, 0, 0, 1]))))

[1, None, 1, None, 1]


**Độ phức tạp:** Time `O(n)`; Space `O(n)` cho `order`, `parent` và hàng đợi. Không đệ quy; ý tưởng "xử lý theo thứ tự ngược tầng" rất hữu ích cho các bài cần thông tin từ con lên cha.

### 7.4. Cách 5 — Dựng cây mới (không sửa cây gốc)

**Giải thích từng đoạn.** Thay vì chỉnh sửa cây có sẵn, hàm trả về **một cây mới**: `new_left = copy_prune(node.left)`, `new_right = copy_prune(node.right)`; nếu `node.val == 0` và hai bản mới đều `None` thì trả `None`; ngược lại tạo `TreeNode(node.val, new_left, new_right)`. Hữu ích khi cần **giữ nguyên cây gốc** (bất biến, immutable) hoặc cần so sánh trước/sau.

In [17]:
def prune_copy(node):
    if node is None:
        return None
    new_left = prune_copy(node.left)
    new_right = prune_copy(node.right)
    if node.val == 0 and new_left is None and new_right is None:
        return None
    return TreeNode(node.val, new_left, new_right)


for values, expected in examples:
    original = build_tree(values)
    got = to_list(prune_copy(original))
    print(values, "->", got, "(kỳ vọng", expected, ") | cây gốc còn nguyên:", to_list(original) == values)
    assert got == expected and to_list(original) == values

[1, None, 0, 0, 1] -> [1, None, 0, None, 1] (kỳ vọng [1, None, 0, None, 1] ) | cây gốc còn nguyên: True
[1, 0, 1, 0, 0, 0, 1] -> [1, None, 1, None, 1] (kỳ vọng [1, None, 1, None, 1] ) | cây gốc còn nguyên: True
[1, 1, 0, 1, 1, 0, 1, 0] -> [1, 1, 0, 1, 1, None, 1] (kỳ vọng [1, 1, 0, 1, 1, None, 1] ) | cây gốc còn nguyên: True


In [18]:
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        def go(node):
            if node is None:
                return None
            left, right = go(node.left), go(node.right)
            if node.val == 0 and left is None and right is None:
                return None
            return TreeNode(node.val, left, right)
        return go(root)

print(to_list(Solution().pruneTree(build_tree([1, 1, 0, 1, 1, 0, 1, 0]))))

[1, 1, 0, 1, 1, None, 1]


**Độ phức tạp:** Time `O(n)`; Space `O(n)` cho cây mới (cộng ngăn xếp đệ quy `O(h)`). Đề LeetCode chấp nhận cả cây mới lẫn cây sửa tại chỗ.

## 8. TRACE từng bước

Trace đệ quy hậu thứ tự (cách 2) với ví dụ 1: `root = [1, None, 0, 0, 1]`. Cây: gốc `A=1`; con phải `B=0`; `B` có con trái `C=0` và con phải `D=1`.

Thứ tự **vào** hàm: `A → B → C → (C.left, C.right rỗng) → D → ...`. Thứ tự **ra** (quyết định): `C, D, B, A`.

| thứ tự quyết định | nút | `val` | con trái sau cắt | con phải sau cắt | quyết định |
|---|---|---|---|---|---|
| 1 | `C` | 0 | `None` | `None` | `val == 0` và không con → **cắt** (trả `None`) |
| 2 | `D` | 1 | `None` | `None` | `val == 1` → giữ |
| 3 | `B` | 0 | `None` (C bị cắt) | `D` | còn con phải → giữ |
| 4 | `A` | 1 | `None` | `B` | `val == 1` → giữ |

Kết quả: `A(1) -> phải B(0) -> phải D(1)`: danh sách `[1, None, 0, None, 1]`.

In [19]:
def trace_prune(root):
    step = [0]

    def go(node, path):
        if node is None:
            return None
        node.left = go(node.left, path + "L")
        node.right = go(node.right, path + "R")
        step[0] += 1
        has_left, has_right = node.left is not None, node.right is not None
        if node.val == 0 and not has_left and not has_right:
            print(f"  #{step[0]} nút '{path or 'gốc'}' val=0, con trái={has_left}, con phải={has_right} -> CẮT")
            return None
        print(f"  #{step[0]} nút '{path or 'gốc'}' val={node.val}, con trái={has_left}, con phải={has_right} -> giữ")
        return node

    print("cây trước:", to_list(root))
    result = go(root, "")
    print("cây sau  :", to_list(result))
    return result

assert to_list(trace_prune(build_tree([1, None, 0, 0, 1]))) == [1, None, 0, None, 1]

cây trước: [1, None, 0, 0, 1]
  #1 nút 'RL' val=0, con trái=False, con phải=False -> CẮT
  #2 nút 'RR' val=1, con trái=False, con phải=False -> giữ
  #3 nút 'R' val=0, con trái=False, con phải=True -> giữ
  #4 nút 'gốc' val=1, con trái=False, con phải=True -> giữ
cây sau  : [1, None, 0, None, 1]


Trace ví dụ 2 `[1, 0, 1, 0, 0, 0, 1]` (gốc `1`; con trái `0` có hai con `0, 0`; con phải `1` có con `0, 1`). Thứ tự quyết định: hai lá `0` của nút trái → cắt; nút `0` trái giờ không con → cắt; lá `0` của nút phải → cắt; lá `1` → giữ; nút phải `1` → giữ; gốc → giữ. Kết quả `[1, None, 1, None, 1]`.

In [20]:
assert to_list(trace_prune(build_tree([1, 0, 1, 0, 0, 0, 1]))) == [1, None, 1, None, 1]
print()
# gốc cũng có thể bị cắt: cây không có số 1 nào
assert to_list(trace_prune(build_tree([0, 0, 0]))) == []

cây trước: [1, 0, 1, 0, 0, 0, 1]
  #1 nút 'LL' val=0, con trái=False, con phải=False -> CẮT
  #2 nút 'LR' val=0, con trái=False, con phải=False -> CẮT
  #3 nút 'L' val=0, con trái=False, con phải=False -> CẮT
  #4 nút 'RL' val=0, con trái=False, con phải=False -> CẮT
  #5 nút 'RR' val=1, con trái=False, con phải=False -> giữ
  #6 nút 'R' val=1, con trái=False, con phải=True -> giữ
  #7 nút 'gốc' val=1, con trái=False, con phải=True -> giữ
cây sau  : [1, None, 1, None, 1]

cây trước: [0, 0, 0]
  #1 nút 'L' val=0, con trái=False, con phải=False -> CẮT
  #2 nút 'R' val=0, con trái=False, con phải=False -> CẮT
  #3 nút 'gốc' val=0, con trái=False, con phải=False -> CẮT
cây sau  : []


Nhận xét từ trace thứ hai: nút `0` ở trên cùng của một dây chuyền chỉ biết nó phải cắt **sau** khi các nút `0` bên dưới đã cắt. Nếu ta kiểm tra "lá `0`" **trước** khi xử lý hai con (tiền thứ tự) thì nút này vẫn còn con tại thời điểm đó và bị giữ nhầm (xem lỗi 1 ở mục 13).

## 9. Code nộp LeetCode

Lời giải đệ quy hậu thứ tự (Cách 2). Trên LeetCode, lớp `TreeNode` do hệ thống cung cấp; ở đây ta đã tự định nghĩa ở mục 2:

In [21]:
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left = self.pruneTree(root.left)          # cắt con trái rồi gán lại
        root.right = self.pruneTree(root.right)        # cắt con phải rồi gán lại
        if root.val == 0 and root.left is None and root.right is None:
            return None                                # lá 0 sau khi cắt: bỏ
        return root


sol = Solution()
for values, expected in examples:
    got = to_list(sol.pruneTree(build_tree(values)))
    print(got)
    assert got == expected

[1, None, 0, None, 1]
[1, None, 1, None, 1]
[1, 1, 0, 1, 1, None, 1]


## 10. Vì sao đúng

**Định nghĩa.** Với nút `v`, gọi `S(v)` là cây con của `v`. `S(v)` bị loại khi nó **không chứa `1`**. Kết quả đúng là cây gồm đúng các nút `v` mà `S(v)` chứa `1` (nếu `S(v)` chứa `1` thì mọi tổ tiên của `v` cũng có cây con chứa `1`, nên tập này đóng kín dưới việc lấy cha: nó là một cây liên thông chứa gốc hoặc rỗng).

**Bổ đề 1 — quy nạp trên chiều cao.** Khẳng định: `prune(v)` trả về `None` nếu `S(v)` không có `1`, ngược lại trả về `v` với hai con là kết quả đúng của việc cắt `S(v.left)`, `S(v.right)`.

- *Cơ sở:* `v is None` trả `None`: đúng (cây rỗng không có `1`).
- *Bước quy nạp:* giả sử đúng cho hai con. Sau khi gán `v.left`, `v.right` bằng kết quả đệ quy: `v.left` là `None` khi `S(v.left)` không có `1`, ngược lại là cây con chứa `1`. Do đó `S(v)` có `1` khi và chỉ khi `v.val == 1` hoặc `v.left is not None` hoặc `v.right is not None` (vì một con còn lại ⇔ cây con của nó có `1`). Vậy nút `v` bị loại ⇔ `v.val == 0` và cả hai con `None`, đúng điều kiện mã.

**Bổ đề 2 — vì sao phải hậu thứ tự.** Quyết định cho `v` phụ thuộc vào việc hai con **còn tồn tại sau khi cắt**; thông tin này chỉ có khi hai con đã xử lý xong. Mọi cách giải đúng (cách 3, 4, 5) đều đảm bảo con trước cha.

**Bổ đề 3 — trả về gốc.** Gốc cũng có thể bị cắt (khi không có `1` nào trong cả cây): hàm trả `None` và LeetCode in `[]`. Đó là lý do hàm trả về gốc mới thay vì sửa tại chỗ.

**Kết luận.** Mọi nút giữ lại có cây con chứa `1`; mọi nút bị cắt có cây con không chứa `1`; các nút giữ lại giữ nguyên quan hệ cha-con; vậy kết quả đúng.

In [22]:
# Kiểm chứng Bổ đề 1 bằng thực nghiệm: "nút còn lại <=> cây con chứa 1" trên nhiều cây ngẫu nhiên
def all_nodes(root):
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        if node is not None:
            out.append(node)
            stack.append(node.left)
            stack.append(node.right)
    return out

def random_values(rng, n_max=25, p_one=0.3):
    n = rng.randint(1, n_max)
    root = TreeNode(1 if rng.random() < p_one else 0)
    free = [(root, "L"), (root, "R")]
    for _ in range(n - 1):
        parent, side = free.pop(rng.randrange(len(free)))
        child = TreeNode(1 if rng.random() < p_one else 0)
        setattr(parent, "left" if side == "L" else "right", child)
        free += [(child, "L"), (child, "R")]
    return to_list(root)

rng = random.Random(10)
for _ in range(500):
    values = random_values(rng, 25, rng.choice([0.1, 0.3, 0.6]))
    before = build_tree(values)
    keep_ids = {id(x) for x in all_nodes(before) if contains_one(x)}      # tập kỳ vọng được giữ
    after = prune_recursive(before)
    kept_ids = {id(x) for x in all_nodes(after)} if after is not None else set()
    assert keep_ids == kept_ids, values
print("Tập nút giữ lại = tập nút có cây con chứa 1 (500 cây ngẫu nhiên)")

Tập nút giữ lại = tập nút có cây con chứa 1 (500 cây ngẫu nhiên)


## 11. Time / Space Complexity

Đặt `n` là số nút, `h` là chiều cao cây (`1 ≤ h ≤ n`).

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra lại `contains_one` cho từng nút | `O(n^2)` (xấu nhất) | `O(h)` |
| 2. Đệ quy hậu thứ tự | `O(n)` | `O(h)` |
| 3. Hậu thứ tự lặp bằng ngăn xếp | `O(n)` | `O(n)` |
| 4. Duyệt tầng ngược + gỡ khỏi cha | `O(n)` | `O(n)` |
| 5. Dựng cây mới | `O(n)` | `O(n)` |

Với `n ≤ 200` mọi cách đều rất nhanh, kể cả brute `O(n^2)`. Ý nghĩa của bài là rèn tư duy **hậu thứ tự** và kỹ thuật **trả về nút để cha gán lại**.

In [23]:
# Đo với cây dây chuyền sâu (xấu nhất cho cách 1) và cây đầy đủ, kích thước tối đa n = 200
def chain_tree(n, last):
    # cây dây chuyền: mỗi nút có con trái duy nhất; nút cuối có giá trị last
    root = TreeNode(0)
    cur = root
    for i in range(1, n):
        cur.left = TreeNode(0 if i < n - 1 else last)
        cur = cur.left
    return root

solvers_tree = {
    "brute": prune_brute, "recursive": prune_recursive, "iterative": prune_iterative,
    "reverse_bfs": prune_reverse_bfs, "copy": prune_copy,
}
for last in (0, 1):
    results = {}
    for name, f in solvers_tree.items():
        t0 = time.perf_counter()
        results[name] = to_list(f(chain_tree(200, last)))
        assert time.perf_counter() - t0 < 5
    assert all(r == results["recursive"] for r in results.values())
    print(f"dây chuyền 200 nút, nút cuối = {last}: kết quả có {len(results['recursive'])} phần tử, 5 cách khớp nhau")

dây chuyền 200 nút, nút cuối = 0: kết quả có 0 phần tử, 5 cách khớp nhau
dây chuyền 200 nút, nút cuối = 1: kết quả có 398 phần tử, 5 cách khớp nhau


## 12. Bộ test

Chia 4 nhóm: (a) ví dụ đề và edge case; (b) phản ví dụ cho các heuristic sai; (c) đối chiếu mọi cách với **oracle độc lập** (xác định nút giữ lại theo "đường đi" từ gốc, không dùng đệ quy cắt); (d) test ngẫu nhiên có seed.

Mọi hàm **sửa cây tại chỗ**, nên mỗi lần gọi ta dựng cây mới từ danh sách bằng `build_tree`.

### 12.1. Oracle độc lập và test cố định

Oracle: mỗi nút được xác định bởi **đường đi** từ gốc (chuỗi `L`/`R`, ví dụ `"LR"`). Một nút giữ lại khi và chỉ khi tồn tại một nút có giá trị `1` có đường đi **bắt đầu bằng** đường đi của nó. Dựng lại danh sách theo tầng từ tập đường đi giữ lại.

In [24]:
def tree_paths(root):
    paths, stack = {}, [(root, "")]
    while stack:
        node, path = stack.pop()
        if node is None:
            continue
        paths[path] = node.val
        stack.append((node.left, path + "L"))
        stack.append((node.right, path + "R"))
    return paths


def oracle_prune(values):
    paths = tree_paths(build_tree(values))
    ones = [p for p, v in paths.items() if v == 1]
    kept = {p for p in paths if any(q.startswith(p) for q in ones)}
    if "" not in kept:
        return []
    out, queue = [], deque([""])
    while queue:
        p = queue.popleft()
        if p is None:
            out.append(None)
            continue
        out.append(paths[p])
        for child in (p + "L", p + "R"):
            queue.append(child if child in kept else None)
    while out and out[-1] is None:
        out.pop()
    return out


solvers = {
    "brute": prune_brute,
    "recursive": prune_recursive,
    "iterative": prune_iterative,
    "reverse_bfs": prune_reverse_bfs,
    "copy": prune_copy,
    "submit": Solution().pruneTree,
}

fixed_cases = [
    ([1, None, 0, 0, 1], [1, None, 0, None, 1]),             # ví dụ 1
    ([1, 0, 1, 0, 0, 0, 1], [1, None, 1, None, 1]),          # ví dụ 2
    ([1, 1, 0, 1, 1, 0, 1, 0], [1, 1, 0, 1, 1, None, 1]),   # ví dụ 3
    ([1], [1]),                                              # một nút 1
    ([0], []),                                               # một nút 0: cả cây bị cắt
    ([0, 0, 0], []),                                         # không có 1
    ([1, 0, 0], [1]),                                        # gốc 1 với hai lá 0
    ([0, 1, 0], [0, 1]),                                     # gốc 0 giữ vì con trái chứa 1
    ([0, None, 0, None, 0, None, 1], [0, None, 0, None, 0, None, 1]),   # dây chuyền phải kết thúc bằng 1
    ([0, None, 0, None, 0, None, 0], []),                    # dây chuyền toàn 0
    ([1, 1, 1], [1, 1, 1]),                                  # toàn 1: không đổi
    ([0, 1, 0, 0, 1, 0, 0], [0, 1, None, None, 1]),          # có nút 0 giữ vì con chứa 1
]
for values, expected in fixed_cases:
    assert oracle_prune(values) == expected, ("oracle", values, oracle_prune(values), expected)
    for name, f in solvers.items():
        got = to_list(f(build_tree(values)))
        assert got == expected, (name, values, got, expected)
print("Test cố định: OK (", len(fixed_cases), "bộ x", len(solvers), "lời giải + oracle )")

Test cố định: OK ( 12 bộ x 6 lời giải + oracle )


### 12.2. Phản ví dụ cho các heuristic sai

- *"Xóa mọi nút có giá trị `0`"* (bỏ qua việc nó có con chứa `1`): cây `[0, 1, 0]` mất cả gốc.
- *"Chỉ xóa các lá `0` một lượt"* (không dây chuyền): dây chuyền `0 → 0 → 0` chỉ mất nút cuối.

In [25]:
def wrong_remove_all_zeros(node):
    if node is None or node.val == 0:
        return None                                  # SAI: bỏ cả cây con dù còn chứa 1
    node.left = wrong_remove_all_zeros(node.left)
    node.right = wrong_remove_all_zeros(node.right)
    return node


def wrong_one_pass_leaves(node):
    # SAI: chỉ xóa các lá 0 của cây ban đầu, không dây chuyền
    if node is None:
        return None
    if node.left is None and node.right is None:
        return None if node.val == 0 else node
    node.left = wrong_one_pass_leaves(node.left)
    node.right = wrong_one_pass_leaves(node.right)
    return node


v1 = [0, 1, 0]
print("xóa mọi 0:", to_list(wrong_remove_all_zeros(build_tree(v1))), "| đúng:", oracle_prune(v1))
assert to_list(wrong_remove_all_zeros(build_tree(v1))) == [] and oracle_prune(v1) == [0, 1]

v2 = [1, 0, None, 0, None, 0]
print("một lượt:", to_list(wrong_one_pass_leaves(build_tree(v2))), "| đúng:", oracle_prune(v2))
assert to_list(wrong_one_pass_leaves(build_tree(v2))) != oracle_prune(v2) == [1]

xóa mọi 0: [] | đúng: [0, 1]
một lượt: [1, 0, None, 0] | đúng: [1]


### 12.3. Đối chiếu oracle trên mọi cây nhỏ

Liệt kê **mọi** cây nhị phân tối đa 7 nút (mọi hình dạng) với **mọi** gán giá trị `0/1`: tổng cộng vài chục nghìn cây.

In [26]:
def all_shapes(n):
    """Mọi hình dạng cây nhị phân có đúng n nút, biểu diễn lồng nhau (None hoặc (trái, phải))."""
    if n == 0:
        return [None]
    out = []
    for k in range(n):
        for left in all_shapes(k):
            for right in all_shapes(n - 1 - k):
                out.append((left, right))
    return out


def shape_to_tree(shape, vals):
    if shape is None:
        return None
    left, right = shape
    node = TreeNode(vals.pop(0))
    node.left = shape_to_tree(left, vals)
    node.right = shape_to_tree(right, vals)
    return node


from itertools import product
checked = 0
for n in range(1, 8):
    for shape in all_shapes(n):
        for vals in product((0, 1), repeat=n):
            values = to_list(shape_to_tree(shape, list(vals)))
            expected = oracle_prune(values)
            for name, f in solvers.items():
                got = to_list(f(build_tree(values)))
                assert got == expected, (name, values, got, expected)
            checked += 1
print("Mọi cây ≤ 7 nút x mọi gán 0/1:", checked, "cây, mọi cách đều khớp oracle")

Mọi cây ≤ 7 nút x mọi gán 0/1: 64978 cây, mọi cách đều khớp oracle


### 12.4. Test ngẫu nhiên có seed

Cây ngẫu nhiên tới 200 nút (đúng ràng buộc) với nhiều xác suất xuất hiện số `1`, kể cả cây không có `1` (kết quả rỗng).

In [27]:
rng = random.Random(814)
empty_results = 0
for _ in range(1500):
    values = random_values(rng, 200, rng.choice([0.0, 0.02, 0.1, 0.3, 0.6]))
    expected = oracle_prune(values)
    empty_results += expected == []
    for name, f in solvers.items():
        got = to_list(f(build_tree(values)))
        assert got == expected, (name, len(values))
print("Ngẫu nhiên (seed 814): 1500 cây tới 200 nút, khớp oracle; kết quả rỗng:", empty_results)

Ngẫu nhiên (seed 814): 1500 cây tới 200 nút, khớp oracle; kết quả rỗng: 395


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quyết định cắt **trước** khi xử lý hai con (tiền thứ tự)

Kiểm tra `val == 0 and không có con` ngay đầu hàm sẽ bỏ sót các nút `0` mà **chỉ sau khi cắt các con** mới trở thành lá. Ví dụ cây `[1, 0, None, 0]` (dây chuyền `1 → 0 → 0` đi xuống bên trái): nút `0` ở giữa còn con `0` tại thời điểm kiểm tra nên bị giữ nhầm, trong khi đúng ra cả hai nút `0` đều phải bị cắt.

In [28]:
def wrong_preorder(node):
    if node is None:
        return None
    if node.val == 0 and node.left is None and node.right is None:    # SAI: kiểm tra trước khi cắt con
        return None
    node.left = wrong_preorder(node.left)
    node.right = wrong_preorder(node.right)
    return node

v = [1, 0, None, 0]
print("sai:", to_list(wrong_preorder(build_tree(v))), "| đúng:", to_list(Solution().pruneTree(build_tree(v))))
assert to_list(wrong_preorder(build_tree(v))) == [1, 0] and to_list(Solution().pruneTree(build_tree(v))) == [1]

sai: [1, 0] | đúng: [1]


### Lỗi 2 — Gọi đệ quy nhưng quên gán lại con

`prune(root.left)` trả về `None` nếu cây con bị cắt, nhưng nếu **không gán** `root.left = ...` thì cây gốc vẫn giữ con cũ: không có gì bị cắt (hàm chỉ trả đúng cho nút gốc).

In [29]:
def wrong_no_assign(node):
    if node is None:
        return None
    wrong_no_assign(node.left)           # SAI: bỏ kết quả, không gán lại
    wrong_no_assign(node.right)
    if node.val == 0 and node.left is None and node.right is None:
        return None
    return node

v = [1, 0, 0]
print("sai:", to_list(wrong_no_assign(build_tree(v))), "| đúng:", to_list(Solution().pruneTree(build_tree(v))))
assert to_list(wrong_no_assign(build_tree(v))) == [1, 0, 0] and to_list(Solution().pruneTree(build_tree(v))) == [1]

sai: [1, 0, 0] | đúng: [1]


### Lỗi 3 — Xóa nút `0` dù con của nó còn chứa `1`

Điều kiện đúng là `val == 0` **và** hai con rỗng. Nếu chỉ kiểm tra `val == 0` rồi trả `None`, ta mất cả những nhánh chứa `1` bên dưới (đã thấy ở mục 12.2).

In [30]:
v = [0, 1, 0]
print("sai (xóa mọi 0):", to_list(wrong_remove_all_zeros(build_tree(v))), "| đúng:", to_list(Solution().pruneTree(build_tree(v))))
assert to_list(Solution().pruneTree(build_tree(v))) == [0, 1]

sai (xóa mọi 0): [] | đúng: [0, 1]


### Lỗi 4 — Chỉ cắt các lá `0` một lượt

Sau khi cắt lá `0`, nút cha (nếu là `0`) trở thành lá `0` và phải bị cắt tiếp; một lượt duyệt "theo cây ban đầu" không lan được lên. Hậu thứ tự làm việc đó tự nhiên (đã thấy ở mục 12.2).

In [31]:
v = [1, 0, None, 0, None, 0]
print("một lượt:", to_list(wrong_one_pass_leaves(build_tree(v))), "| đúng:", to_list(Solution().pruneTree(build_tree(v))))

một lượt: [1, 0, None, 0] | đúng: [1]


### Lỗi 5 — Quên rằng gốc cũng có thể bị cắt

Nếu viết `prune(root.left); prune(root.right); return root` thì cây `[0, 0, 0]` trả về gốc `0` thay vì cây rỗng. Hàm phải **trả về kết quả của việc xử lý gốc** (có thể `None`).

In [32]:
def wrong_always_return_root(root):
    def go(node):
        if node is None:
            return None
        node.left = go(node.left)
        node.right = go(node.right)
        if node.val == 0 and node.left is None and node.right is None:
            return None
        return node
    go(root)
    return root                          # SAI: gốc có thể đã bị cắt nhưng ta trả lại gốc cũ

print("sai:", to_list(wrong_always_return_root(build_tree([0, 0, 0]))), "| đúng:", to_list(Solution().pruneTree(build_tree([0, 0, 0]))))
assert to_list(wrong_always_return_root(build_tree([0, 0, 0]))) == [0] and to_list(Solution().pruneTree(build_tree([0, 0, 0]))) == []

sai: [0] | đúng: []


## 14. Mẹo & nhận dạng pattern

- **"Xóa/cắt các cây con thỏa điều kiện phụ thuộc vào cả cây con"** → đệ quy **hậu thứ tự** và **trả về nút (hoặc `None`)** để cha gán lại: `node.left = f(node.left)`.
- **Quyết định phụ thuộc vào kết quả của hai con** (có `1` không, tổng, độ sâu...) → hậu thứ tự; **phụ thuộc vào cha** (đường đi từ gốc xuống) → tiền thứ tự.
- Điều kiện cắt thường có dạng "**giá trị đặc biệt và cả hai con đều rỗng**", sau khi các con đã được xử lý.
- **Cây rất sâu** (vượt giới hạn đệ quy) → ngăn xếp tường minh hoặc duyệt tầng ngược (cách 3, 4).
- **Cần giữ cây gốc** → dựng cây mới (cách 5).
- **Dựng/xuất cây** từ danh sách theo thứ tự tầng: nhớ nút `None` không chiếm chỗ cho con; bỏ `None` thừa ở cuối khi xuất.
- Các bài cùng họ: Delete Leaves With a Given Value (1325), Delete Nodes And Return Forest (1110), Maximum Depth of Binary Tree (104), Invert Binary Tree (226), Trim a Binary Search Tree (669).

## 15. Biến thể

### 15.1. Cắt các cây con không chứa giá trị `x` bất kỳ

Thay `1` bằng giá trị `x` tuỳ ý và giá trị nút tuỳ ý: nút bị cắt khi `val != x` và hai con đều rỗng.

In [33]:
def prune_without(root, x):
    if root is None:
        return None
    root.left = prune_without(root.left, x)
    root.right = prune_without(root.right, x)
    if root.val != x and root.left is None and root.right is None:
        return None
    return root

print(to_list(prune_without(build_tree([3, 1, 2, 5, 1, 2, 4]), 4)))   # chỉ giữ đường tới nút 4
print(to_list(prune_without(build_tree([1, None, 0, 0, 1]), 1)))
assert to_list(prune_without(build_tree([1, None, 0, 0, 1]), 1)) == [1, None, 0, None, 1]

[3, None, 2, None, 4]
[1, None, 0, None, 1]


### 15.2. Xóa lá có giá trị `target` lặp lại (LeetCode 1325)

Xóa các lá có `val == target`; sau khi xóa, nút cha có thể thành lá `target` và bị xóa tiếp. Với cây `0/1` và `target = 0` kết quả **trùng** bài 814.

In [34]:
def delete_leaves(root, target):
    if root is None:
        return None
    root.left = delete_leaves(root.left, target)
    root.right = delete_leaves(root.right, target)
    if root.left is None and root.right is None and root.val == target:
        return None
    return root

print(to_list(delete_leaves(build_tree([1, 2, 3, 2, None, 2, 4]), 2)))     # [1, None, 3, None, 4]
rng = random.Random(15)
for _ in range(500):
    values = random_values(rng, 30, rng.choice([0.2, 0.5]))
    assert to_list(delete_leaves(build_tree(values), 0)) == oracle_prune(values)
print("OK: delete_leaves(target=0) khớp bài 814 trên 500 cây ngẫu nhiên")

[1, None, 3, None, 4]
OK: delete_leaves(target=0) khớp bài 814 trên 500 cây ngẫu nhiên


### 15.3. Các biến thể khác (chỉ mô tả)

- **Cây N-phân** (mỗi nút có danh sách con): lọc danh sách con bằng kết quả đệ quy, nút bị cắt khi `val == 0` và danh sách con rỗng.
- **Đếm số nút bị cắt**: hậu thứ tự trả về thêm số nút đã cắt (bài tập 5 ở mục 17).
- **Cắt khi tổng giá trị cây con bằng 0 / nhỏ hơn ngưỡng**: hậu thứ tự trả về `(nút, tổng)`; cắt theo tổng (xem Delete Nodes And Return Forest, Binary Tree Pruning with sum).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra lại `contains_one` | mỗi nút duyệt cả cây con để biết có `1` | `O(n^2)` | `O(h)` | để thấy nút thắt; làm oracle |
| 2. Đệ quy hậu thứ tự | cắt hai con, rồi quyết định nút | `O(n)` | `O(h)` | **cách nên nộp**: ngắn, chuẩn |
| 3. Hậu thứ tự lặp | ngăn xếp + tập `dead` | `O(n)` | `O(n)` | cây rất sâu (tránh giới hạn đệ quy) |
| 4. Duyệt tầng ngược | xử lý nút sâu trước, tự gỡ khỏi cha | `O(n)` | `O(n)` | khi quen BFS; không đệ quy |
| 5. Dựng cây mới | trả về bản sao đã cắt | `O(n)` | `O(n)` | cần giữ nguyên cây gốc |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(to_list(build_tree([1, 2, 3])))
print(to_list(build_tree([1, None, 2, 3])))
print(to_list(Solution().pruneTree(build_tree([0, 0, 1]))))
```

In [35]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `root = [0, 1, 0, 0, 1, 0, 0]` (gốc `0`; con trái `1` có hai con `0, 1`; con phải `0` có hai con `0, 0`): lần lượt cho biết nút nào bị cắt theo thứ tự hậu thứ tự, và viết danh sách kết quả.

In [36]:
# Bài 2: điền kết quả tính tay
hand_answer = None      # danh sách kết quả kiểu LeetCode, ví dụ [1, None, 1]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm đệ quy hậu thứ tự.

In [37]:
def exercise_3_prune(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO 1: nếu root là None thì trả None
    # TODO 2: cắt con trái và con phải, GÁN LẠI vào root.left, root.right
    # TODO 3: nếu root.val == 0 và cả hai con đều None thì trả None
    # TODO 4: ngược lại trả root
    return None


# assert to_list(exercise_3_prune(build_tree([1, None, 0, 0, 1]))) == [1, None, 0, None, 1]
# assert to_list(exercise_3_prune(build_tree([0, 0, 0]))) == []
# assert to_list(exercise_3_prune(build_tree([1, 1, 0, 1, 1, 0, 1, 0]))) == [1, 1, 0, 1, 1, None, 1]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 3, hãy tự viết `exercise_4_iterative(root)`: hậu thứ tự **lặp** bằng ngăn xếp (hoặc duyệt tầng ngược), không đệ quy.

In [38]:
def exercise_4_iterative(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# assert to_list(exercise_4_iterative(build_tree([1, 0, 1, 0, 0, 0, 1]))) == [1, None, 1, None, 1]
# assert to_list(exercise_4_iterative(build_tree([0]))) == []
# assert to_list(exercise_4_iterative(build_tree([1, 1, 1]))) == [1, 1, 1]

### Bài 5 — Mini challenge: cắt và đếm số nút bị cắt (khó)

Viết `exercise_5_prune_count(root)` trả về cặp `(gốc mới, số nút bị cắt)`. Gợi ý: hàm đệ quy trả thêm số nút bị cắt ở cây con. Vì một nút chỉ bị cắt khi hai con đã bị cắt hết (nên các nút bên dưới đã được tính từ trước), mỗi nút bị cắt chỉ cần cộng thêm `1` đúng lúc nó bị cắt. Kiểm tra bằng `số nút ban đầu - số nút còn lại`.

```python
exercise_5_prune_count(build_tree([1, None, 0, 0, 1])) -> (cây [1,None,0,None,1], 1)
exercise_5_prune_count(build_tree([0, 0, 0])) -> (None, 3)
exercise_5_prune_count(build_tree([1, 1, 1])) -> (cây [1,1,1], 0)
```

In [39]:
def exercise_5_prune_count(root: Optional[TreeNode]):
    # TODO
    pass


# new_root, removed = exercise_5_prune_count(build_tree([1, None, 0, 0, 1]))
# assert to_list(new_root) == [1, None, 0, None, 1] and removed == 1
# new_root, removed = exercise_5_prune_count(build_tree([0, 0, 0]))
# assert new_root is None and removed == 3
# new_root, removed = exercise_5_prune_count(build_tree([1, 1, 1]))
# assert to_list(new_root) == [1, 1, 1] and removed == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: danh sách theo tầng: gốc, rồi hai con của gốc, rồi các con của từng nút (bỏ qua nút `None`).
- Mức 2: `[1, None, 2, 3]` là gốc `1`, con trái vắng, con phải `2`; `2` có con trái `3`.
- Mức 3: dòng 1 `[1, 2, 3]`; dòng 2 `[1, None, 2, 3]`; dòng 3: gốc `0` có con trái `0` (bị cắt) và con phải `1` → `[0, None, 1]`.

**Bài 2**
- Mức 1: duyệt dưới lên: các lá `0, 1` (con của nút trái), `0, 0` (con của nút phải).
- Mức 2: nút trái `1` giữ; con trái của nó (`0`) cắt, con phải (`1`) giữ. Nút phải `0` có hai lá `0` đều bị cắt rồi chính nó bị cắt.
- Mức 3: kết quả: gốc `0` có con trái `1` (với con phải `1`), không con phải: `[0, 1, None, None, 1]`.

**Bài 3**
- Mức 1: giống code mục 9.
- Mức 2: `root.left = exercise_3_prune(root.left)` (gán lại, đừng bỏ kết quả).
- Mức 3: `if root.val == 0 and root.left is None and root.right is None: return None`; cuối `return root`.

**Bài 4**
- Mức 1: ngăn xếp các cặp `(nút, đã_thăm_con)` hoặc `order` BFS rồi duyệt ngược.
- Mức 2: duyệt ngược `order`; với mỗi nút có `val == 0` và không con thì gỡ khỏi cha (cần bảng `parent`).
- Mức 3: nếu nút bị gỡ là gốc thì trả `None`; nếu không, trả `root`.

**Bài 5**
- Mức 1: hàm con trả `(nút, số_cắt)`.
- Mức 2: số nút bị cắt = số nút cây ban đầu trừ số nút cây kết quả: đếm bằng `count_nodes`.
- Mức 3: cách trực tiếp: `go(node)` trả `(node_mới, cắt)`; `cắt = cắt_trái + cắt_phải` và nếu nút này cũng bị cắt thì `+1`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
print(to_list(build_tree([1, 2, 3])))
print(to_list(build_tree([1, None, 2, 3])))
print(to_list(Solution().pruneTree(build_tree([0, 0, 1]))))

assert to_list(build_tree([1, 2, 3])) == [1, 2, 3]
assert to_list(build_tree([1, None, 2, 3])) == [1, None, 2, 3]
assert to_list(Solution().pruneTree(build_tree([0, 0, 1]))) == [0, None, 1]
print("Bài 1 OK")

[1, 2, 3]
[1, None, 2, 3]
[0, None, 1]
Bài 1 OK


### Đáp án Bài 2

Thứ tự hậu thứ tự: con trái của nút trái (`0`) → cắt; con phải của nút trái (`1`) → giữ; nút trái (`1`) → giữ; hai lá `0` của nút phải → cắt; nút phải (`0`, không còn con) → cắt; gốc (`0`, còn con trái) → giữ. Kết quả `[0, 1, None, None, 1]`.

In [41]:
values2 = [0, 1, 0, 0, 1, 0, 0]
ans2 = to_list(Solution().pruneTree(build_tree(values2)))
print(ans2)
assert ans2 == [0, 1, None, None, 1]
assert ans2 == oracle_prune(values2)
print("Bài 2 OK")

[0, 1, None, None, 1]
Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_solution(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if root is None:
        return None
    root.left = exercise_3_solution(root.left)
    root.right = exercise_3_solution(root.right)
    if root.val == 0 and root.left is None and root.right is None:
        return None
    return root


assert to_list(exercise_3_solution(build_tree([1, None, 0, 0, 1]))) == [1, None, 0, None, 1]
assert to_list(exercise_3_solution(build_tree([0, 0, 0]))) == []
assert to_list(exercise_3_solution(build_tree([1, 1, 0, 1, 1, 0, 1, 0]))) == [1, 1, 0, 1, 1, None, 1]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if root is None:
        return None
    order, parent = [], {}
    queue = deque([root])
    while queue:
        node = queue.popleft()
        order.append(node)
        for child, side in ((node.left, "L"), (node.right, "R")):
            if child is not None:
                parent[child] = (node, side)
                queue.append(child)
    for node in reversed(order):
        if node.val == 0 and node.left is None and node.right is None:
            if node is root:
                return None
            par, side = parent[node]
            if side == "L":
                par.left = None
            else:
                par.right = None
    return root


assert to_list(exercise_4_solution(build_tree([1, 0, 1, 0, 0, 0, 1]))) == [1, None, 1, None, 1]
assert to_list(exercise_4_solution(build_tree([0]))) == []
assert to_list(exercise_4_solution(build_tree([1, 1, 1]))) == [1, 1, 1]
rng = random.Random(4)
for _ in range(500):
    values = random_values(rng, 40, rng.choice([0.05, 0.3]))
    assert to_list(exercise_4_solution(build_tree(values))) == oracle_prune(values)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_solution(root: Optional[TreeNode]):
    def go(node):
        if node is None:
            return None, 0
        node.left, cut_left = go(node.left)
        node.right, cut_right = go(node.right)
        cut = cut_left + cut_right
        if node.val == 0 and node.left is None and node.right is None:
            return None, cut + 1
        return node, cut
    return go(root)


new_root, removed = exercise_5_solution(build_tree([1, None, 0, 0, 1]))
assert to_list(new_root) == [1, None, 0, None, 1] and removed == 1
new_root, removed = exercise_5_solution(build_tree([0, 0, 0]))
assert new_root is None and removed == 3
new_root, removed = exercise_5_solution(build_tree([1, 1, 1]))
assert to_list(new_root) == [1, 1, 1] and removed == 0
rng = random.Random(5)
for _ in range(500):
    values = random_values(rng, 40, rng.choice([0.0, 0.05, 0.3]))
    before = count_nodes(build_tree(values))
    new_root, removed = exercise_5_solution(build_tree(values))
    assert before - count_nodes(new_root) == removed
    assert to_list(new_root) == oracle_prune(values)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left = self.pruneTree(root.left)           # cắt con trái, gán lại
        root.right = self.pruneTree(root.right)         # cắt con phải, gán lại
        if root.val == 0 and root.left is None and root.right is None:
            return None                                 # lá 0 sau khi cắt: bỏ
        return root
```

Câu thần chú: **"cắt hai con trước (và gán lại), rồi nút nào là `0` mà không còn con thì bỏ."**

### Checklist trước khi submit

- [ ] Xử lý hai con **trước**, quyết định cho nút **sau** (hậu thứ tự).
- [ ] **Gán lại** `root.left = ...`, `root.right = ...` (không bỏ kết quả đệ quy).
- [ ] Điều kiện bỏ nút: `val == 0` **và** `left is None` **và** `right is None`.
- [ ] Trả về gốc mới (có thể là `None` khi cả cây không có `1`).
- [ ] Dùng `is None` (không dùng `== None`).
- [ ] Test: một nút `0`, một nút `1`, cây toàn `0`, cây toàn `1`, dây chuyền dài, ba ví dụ đề.
- [ ] Skeleton đúng: `class Solution`, method `pruneTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]`.